In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_excel('附件：数据.xlsx', header=0)
df
# 读取所有工作表
all_sheets = pd.read_excel('附件：数据.xlsx', sheet_name=None)
all_sheets


{'附件1':         地区       石家庄         唐山       秦皇岛         邯郸         邢台         保定  \
 0  需求量（万吨）  31.53822  30.175434  9.065831  36.977719  36.911715  43.383981   
 
          张家口         承德         沧州         廊坊         衡水    北京市   天津市  
 0  38.169672  14.959592  39.912952  20.309791  30.385093  10.53  6.34  ,
 '附件2':      单位:公里    石家庄     唐山    秦皇岛     邯郸     邢台     保定    张家口     承德     沧州  \
 0  仓库A(玉田)  410.2   63.7  177.6  562.2  516.3  275.4  312.3  162.0  246.0   
 1  仓库B(辛集)   80.8  385.3  530.2  213.2  167.3  149.6  439.1  513.4  168.2   
 2  仓库C(南宫)  124.4  428.6  573.6  150.3  104.4  222.5  518.8  556.5  211.5   
 
       廊坊     衡水     北京     天津  
 0  138.0  376.0  135.9  123.2  
 1  272.9   70.6  292.0  271.7  
 2  315.9   55.0  335.1  315.1  ,
 '附件3':   单位:万吨  仓库A(玉田)  仓库B(辛集)  仓库C(南宫)
 0  化肥存量       85      120      105,
 '附件4':     车型  最大运量  单位交通成本（元/公里/吨）  单位油耗（元/公里）
 0  小型车    10          0.1450         1.4
 1  中型车    20          0.0975         3.6
 2  重型车    40      

In [6]:
from scipy.optimize import linprog
import numpy as np

# 各仓库的化肥实际存量（万吨）
warehouse_stocks = np.array([85, 120, 105])

# 京津冀地区的化肥需求量（万吨）
city_demands = np.array([31.538220487736385, 30.175433907507962, 9.065830953941212, 36.97771906010111, 36.911715151657, 43.38398076202959, 38.16967199494477, 14.959591719715412, 39.91295169443925, 20.309790886538106, 30.38509338138926, 10.53, 6.34])

# 各仓库与配送目标间的公路距离（公里）
distances = np.array([
    [410.2, 63.7, 177.6, 562.2, 516.3, 275.4, 312.3, 162.0, 246.0, 138.0, 376.0, 135.9, 123.2],
    [80.8, 385.3, 530.2, 213.2, 167.3, 149.6, 439.1, 513.4, 168.2, 272.9, 70.6, 292.0, 271.7],
    [124.4, 428.6, 573.6, 150.3, 104.4, 222.5, 518.8, 556.5, 211.5, 315.9, 55.0, 335.1, 315.1]
])

# 运输成本为0.10元/公里/吨
transport_cost = 0.10

# 构建目标函数系数，目标是最大化满足的需求总量
c = -np.ones(len(city_demands) * len(warehouse_stocks))

# 构建不等式约束矩阵 A_ub 和向量 b_ub（仓库库存限制）
A_ub = []
b_ub = []
num_cities = len(city_demands)
for i in range(len(warehouse_stocks)):
    row = [0] * num_cities * len(warehouse_stocks)
    for j in range(num_cities):
        row[i * num_cities + j] = 1
    A_ub.append(row)
    b_ub.append(warehouse_stocks[i])

# 构建不等式约束矩阵 A_ub2 和向量 b_ub2（每个城市分配量不能超过需求量）
A_ub2 = []
b_ub2 = []
for j in range(num_cities):
    row = [0] * num_cities * len(warehouse_stocks)
    for i in range(len(warehouse_stocks)):
        row[i * num_cities + j] = 1
    A_ub2.append(row)
    b_ub2.append(city_demands[j])

# 合并不等式约束
A_ub = np.vstack([A_ub, A_ub2])
b_ub = np.concatenate([b_ub, b_ub2])

# 求解线性规划问题
res = linprog(c, A_ub=A_ub, b_ub=b_ub, method='highs', bounds=(0, None))

if res.status == 0:
    # 提取结果并输出
    optimal_solution = res.x.reshape(len(warehouse_stocks), len(city_demands))
    print('最优配送方案（万吨）：')
    print(optimal_solution)
    print('最小总运输成本（元）：', np.sum(optimal_solution * distances * transport_cost))

    # 计算各地区需求满足率
    demand_satisfaction = np.sum(optimal_solution, axis=0) / city_demands
    print('各地区需求满足率：', demand_satisfaction)
    print('平均需求满足率：', np.mean(demand_satisfaction))
else:
    print(f'线性规划求解失败，状态码: {res.status}')
    if res.status == 1:
        print('超过了迭代次数限制')
    elif res.status == 2:
        print('问题不可行')
    elif res.status == 3:
        print('目标函数无界')
    elif res.status == 4:
        print('算法特定的错误')


最优配送方案（万吨）：
[[ 0.          0.          0.         36.97771906 27.71249005  0.
   0.          0.          0.         20.30979089  0.          0.
   0.        ]
 [ 0.          0.          9.06583095  0.          0.          0.
  38.16967199 14.95959172 39.91295169  0.         17.89195364  0.
   0.        ]
 [31.53822049  0.          0.          0.          7.05465901 43.38398076
   0.          0.          0.          0.         12.49313974 10.53
   0.        ]]
最小总运输成本（元）： 9365.189718229267
各地区需求满足率： [1.         0.         1.         1.         0.94190012 1.
 1.         1.         1.         1.         1.         1.
 0.        ]
平均需求满足率： 0.8416846248829948


In [9]:
import numpy as np
from scipy.optimize import linprog

# 各仓库的化肥实际存量（万吨）
warehouse_stocks = np.array([85, 120, 105])

# 京津冀地区的化肥需求量（万吨）
city_demands = np.array([31.538220487736385, 30.175433907507962, 9.065830953941212, 36.97771906010111, 36.911715151657, 43.38398076202959, 38.16967199494477, 14.959591719715412, 39.91295169443925, 20.309790886538106, 30.38509338138926, 10.53, 6.34])

# 各仓库与配送目标间的公路距离（公里）
distances = np.array([
    [410.2, 63.7, 177.6, 562.2, 516.3, 275.4, 312.3, 162.0, 246.0, 138.0, 376.0, 135.9, 123.2],
    [80.8, 385.3, 530.2, 213.2, 167.3, 149.6, 439.1, 513.4, 168.2, 272.9, 70.6, 292.0, 271.7],
    [124.4, 428.6, 573.6, 150.3, 104.4, 222.5, 518.8, 556.5, 211.5, 315.9, 55.0, 335.1, 315.1]
])

# 运输成本为0.10元/公里/吨
transport_cost = 0.10

# 构建目标函数系数，目标是最小化总运输成本
c = np.array([dist * transport_cost for sublist in distances for dist in sublist])

# 构建不等式约束矩阵 A_ub 和向量 b_ub（每个仓库发货量不超过库存）
A_ub = []
b_ub = []
num_cities = len(city_demands)
for i in range(len(warehouse_stocks)):
    row = [0] * num_cities * len(warehouse_stocks)
    for j in range(num_cities):
        row[i * num_cities + j] = 1
    A_ub.append(row)
    b_ub.append(warehouse_stocks[i])

# 构建不等式约束确保每个地区分配量不超过需求量
for j in range(num_cities):
    row = [0] * num_cities * len(warehouse_stocks)
    for i in range(len(warehouse_stocks)):
        row[i * num_cities + j] = 1
    A_ub.append(row)
    b_ub.append(city_demands[j])

# 构建等式约束矩阵 A_eq 和向量 b_eq（所有库存都要分配出去）
A_eq = []
b_eq = []
for i in range(len(warehouse_stocks)):
    row = [0] * num_cities * len(warehouse_stocks)
    for j in range(num_cities):
        row[i * num_cities + j] = 1
    A_eq.append(row)
b_eq = warehouse_stocks

# 求解线性规划问题
res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, method='highs', bounds=(0, None))

if res.status == 0:
    # 提取结果并输出
    optimal_solution = res.x.reshape(len(warehouse_stocks), len(city_demands))
    print('最优配送方案（万吨）：')
    print(optimal_solution)
    print('最小总运输成本（元）：', np.sum(optimal_solution * distances * transport_cost))

    # 计算各地区需求满足率
    demand_satisfaction = np.sum(optimal_solution, axis=0) / city_demands
    print('各地区需求满足率：', demand_satisfaction)
    print('平均需求满足率：', np.mean(demand_satisfaction))
else:
    print(f'线性规划求解失败，状态码: {res.status}')
    if res.status == 1:
        print('超过了迭代次数限制')
    elif res.status == 2:
        print('问题不可行')
    elif res.status == 3:
        print('目标函数无界')
    elif res.status == 4:
        print('算法特定的错误')


最优配送方案（万吨）：
[[ 0.         30.17543391  8.57550295  0.          0.          0.
   0.         14.95959172  0.         14.41947142  0.         10.53
   6.34      ]
 [31.53822049  0.          0.          0.          0.         43.38398076
   0.          0.         39.91295169  5.16484706  0.          0.
   0.        ]
 [ 0.          0.          0.         36.97771906 36.91171515  0.
   0.          0.          0.          0.72547241 30.38509338  0.
   0.        ]]
最小总运输成本（元）： 3854.370845852836
各地区需求满足率： [1.         1.         0.94591472 1.         1.         1.
 0.         1.         1.         1.         1.         1.
 1.        ]
平均需求满足率： 0.9189165169275


In [7]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# 各仓库的化肥实际存量（万吨）
warehouse_stocks = np.array([85, 120, 105])

# 京津冀地区的化肥需求量（万吨）
city_demands = np.array([31.538220487736385, 30.175433907507962, 9.065830953941212, 36.97771906010111, 36.911715151657, 43.38398076202959, 38.16967199494477, 14.959591719715412, 39.91295169443925, 20.309790886538106, 30.38509338138926, 10.53, 6.34])

# 各仓库与配送目标间的公路距离（公里）
distances = np.array([
    [410.2, 63.7, 177.6, 562.2, 516.3, 275.4, 312.3, 162.0, 246.0, 138.0, 376.0, 135.9, 123.2],
    [80.8, 385.3, 530.2, 213.2, 167.3, 149.6, 439.1, 513.4, 168.2, 272.9, 70.6, 292.0, 271.7],
    [124.4, 428.6, 573.6, 150.3, 104.4, 222.5, 518.8, 556.5, 211.5, 315.9, 55.0, 335.1, 315.1]
])

# 运输成本为0.10元/公里/吨
transport_cost = 0.10

# 找到需求量最小的6个地区的索引
sorted_indices = np.argsort(city_demands)[:6]

# 构建目标函数系数，对需求量最小的6个地区赋予高优先级
c_transport = np.array([dist * transport_cost for sublist in distances for dist in sublist])
priority_weight = 1000  # 优先级权重，可以根据实际情况调整
c_priority = np.zeros(len(c_transport))
for index in sorted_indices:
    for i in range(len(warehouse_stocks)):
        c_priority[i * len(city_demands) + index] = -priority_weight
c = c_transport + c_priority

# 构建不等式约束矩阵 A_ub 和向量 b_ub（每个仓库发货量不超过库存）
A_ub = []
b_ub = []
num_cities = len(city_demands)
for i in range(len(warehouse_stocks)):
    row = [0] * num_cities * len(warehouse_stocks)
    for j in range(num_cities):
        row[i * num_cities + j] = 1
    A_ub.append(row)
    b_ub.append(warehouse_stocks[i])

# 构建不等式约束确保每个地区分配量不超过需求量
for j in range(num_cities):
    row = [0] * num_cities * len(warehouse_stocks)
    for i in range(len(warehouse_stocks)):
        row[i * num_cities + j] = 1
    A_ub.append(row)
    b_ub.append(city_demands[j])

# 构建等式约束矩阵 A_eq 和向量 b_eq（所有库存都要分配出去）
A_eq = []
b_eq = []
for i in range(len(warehouse_stocks)):
    row = [0] * num_cities * len(warehouse_stocks)
    for j in range(num_cities):
        row[i * num_cities + j] = 1
    A_eq.append(row)
b_eq = warehouse_stocks

# 求解线性规划问题
res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, method='highs', bounds=(0, None))

if res.status == 0:
    # 提取结果并输出
    optimal_solution = res.x.reshape(len(warehouse_stocks), len(city_demands))
    print('最优配送方案（万吨）：')
    print(optimal_solution)
    print('最小总运输成本（元）：', np.sum(optimal_solution * distances * transport_cost))

    # 计算各地区需求满足率
    demand_satisfaction = np.sum(optimal_solution, axis=0) / city_demands
    print('各地区需求满足率：', demand_satisfaction)
    print('平均需求满足率：', np.mean(demand_satisfaction))
else:
    print(f'线性规划求解失败，状态码: {res.status}')
    if res.status == 1:
        print('超过了迭代次数限制')
    elif res.status == 2:
        print('问题不可行')
    elif res.status == 3:
        print('目标函数无界')
    elif res.status == 4:
        print('算法特定的错误')
# 读取数据
demand_df = pd.read_excel('附件：数据.xlsx', sheet_name='附件1', index_col=0).squeeze()
distance_df = pd.read_excel('附件：数据.xlsx', sheet_name='附件2', index_col=0)
stock_df = pd.read_excel('附件：数据.xlsx', sheet_name='附件3', index_col=0).squeeze()

# 提取数据
regions = demand_df.index
warehouses = stock_df.index
demand = demand_df.values
distance = distance_df.values
stock = stock_df.values
df = pd.DataFrame(optimal_solution, columns=regions, index=warehouses)
df.to_excel('最优配送方案.xlsx')

最优配送方案（万吨）：
[[ 0.         30.17543391  9.06583095  0.          0.          0.
   0.         14.95959172  0.         13.92914342  0.         10.53
   6.34      ]
 [31.53822049  0.          0.          0.          0.         43.38398076
   0.          0.         39.42262369  5.65517506  0.          0.
   0.        ]
 [ 0.          0.          0.         36.97771906 36.91171515  0.
   0.          0.          0.          0.72547241 30.38509338  0.
   0.        ]]
最小总运输成本（元）： 3861.446278965784
各地区需求满足率： [1.         1.         1.         1.         1.         1.
 0.         1.         0.98771507 1.         1.         1.
 1.        ]
平均需求满足率： 0.9221319281003761


In [39]:
import numpy as np
from scipy.optimize import linprog

# 各仓库的化肥实际存量（万吨）
warehouse_stocks = np.array([85, 120, 105])

# 京津冀地区的化肥需求量（万吨）
city_demands = np.array([31.538220487736385, 30.175433907507962, 9.065830953941212, 36.97771906010111, 36.911715151657, 43.38398076202959, 38.16967199494477, 14.959591719715412, 39.91295169443925, 20.309790886538106, 30.38509338138926, 10.53, 6.34])

# 各仓库与配送目标间的公路距离（公里）
distances = np.array([
    [410.2, 63.7, 177.6, 562.2, 516.3, 275.4, 312.3, 162.0, 246.0, 138.0, 376.0, 135.9, 123.2],
    [80.8, 385.3, 530.2, 213.2, 167.3, 149.6, 439.1, 513.4, 168.2, 272.9, 70.6, 292.0, 271.7],
    [124.4, 428.6, 573.6, 150.3, 104.4, 222.5, 518.8, 556.5, 211.5, 315.9, 55.0, 335.1, 315.1]
])

# 运输成本为0.10元/公里/吨
transport_cost = 0.10

# 找到需求量最小的k个地区的索引
for k in range(1, 14):
    sorted_indices = np.argsort(city_demands)[:k]

    # 构建目标函数系数，对需求量最小的k个地区赋予高优先级
    c_transport = np.array([dist * transport_cost for sublist in distances for dist in sublist])
    priority_weight = 1000  # 优先级权重，可以根据实际情况调整
    c_priority = np.zeros(len(c_transport))
    for index in sorted_indices:
        for i in range(len(warehouse_stocks)):
            c_priority[i * len(city_demands) + index] = -priority_weight
    c = c_transport + c_priority

    # 构建不等式约束矩阵 A_ub 和向量 b_ub（每个仓库发货量不超过库存）
    A_ub = []
    b_ub = []
    num_cities = len(city_demands)
    for i in range(len(warehouse_stocks)):
        row = [0] * num_cities * len(warehouse_stocks)
        for j in range(num_cities):
            row[i * num_cities + j] = 1
        A_ub.append(row)
        b_ub.append(warehouse_stocks[i])

    # 构建不等式约束确保每个地区分配量不超过需求量
    for j in range(num_cities):
        row = [0] * num_cities * len(warehouse_stocks)
        for i in range(len(warehouse_stocks)):
            row[i * num_cities + j] = 1
        A_ub.append(row)
        b_ub.append(city_demands[j])

    # 构建等式约束矩阵 A_eq 和向量 b_eq（所有库存都要分配出去）
    A_eq = []
    b_eq = []
    for i in range(len(warehouse_stocks)):
        row = [0] * num_cities * len(warehouse_stocks)
        for j in range(num_cities):
            row[i * num_cities + j] = 1
        A_eq.append(row)
    b_eq = warehouse_stocks

    # 求解线性规划问题
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, method='highs', bounds=(0, None))

    if res.status == 0:
        # 提取结果并输出
        optimal_solution = res.x.reshape(len(warehouse_stocks), len(city_demands))
        # print('最优配送方案（万吨）：')
        # print(optimal_solution)
        print(f'优先满足需求量最小的{k}个地区：')
        print('最小总运输成本（元）：', np.sum(optimal_solution * distances * transport_cost))

        # 计算各地区需求满足率
        demand_satisfaction = np.sum(optimal_solution, axis=0) / city_demands
        # print('各地区需求满足率：', demand_satisfaction)
        print('平均需求满足率：', np.mean(demand_satisfaction))
    else:
        print(f'线性规划求解失败，状态码: {res.status}')
        if res.status == 1:
            print('超过了迭代次数限制')
        elif res.status == 2:
            print('问题不可行')
        elif res.status == 3:
            print('目标函数无界')
        elif res.status == 4:
            print('算法特定的错误')


优先满足需求量最小的1个地区：
最小总运输成本（元）： 3854.370845852836
平均需求满足率： 0.9189165169275
优先满足需求量最小的2个地区：
最小总运输成本（元）： 3855.135757540723
平均需求满足率： 0.9205556284081188
优先满足需求量最小的3个地区：
最小总运输成本（元）： 3855.135757540723
平均需求满足率： 0.9205556284081188
优先满足需求量最小的4个地区：
最小总运输成本（元）： 3856.312544752855
平均需求满足率： 0.921219811939683
优先满足需求量最小的5个地区：
最小总运输成本（元）： 3859.955681830416
平均需求满足率： 0.9218269811882325
优先满足需求量最小的6个地区：
最小总运输成本（元）： 3861.446278965784
平均需求满足率： 0.9221319281003761
优先满足需求量最小的7个地区：
最小总运输成本（元）： 3861.446278965784
平均需求满足率： 0.9221319281003761
优先满足需求量最小的8个地区：
最小总运输成本（元）： 3861.446278965784
平均需求满足率： 0.9221319281003761
优先满足需求量最小的9个地区：
最小总运输成本（元）： 3861.446278965784
平均需求满足率： 0.9221319281003761
优先满足需求量最小的10个地区：
最小总运输成本（元）： 3861.446278965784
平均需求满足率： 0.9221319281003761
优先满足需求量最小的11个地区：
最小总运输成本（元）： 4895.462693308838
平均需求满足率： 0.9254917006235728
优先满足需求量最小的12个地区：
最小总运输成本（元）： 4967.370293308839
平均需求满足率： 0.9314528980141693
优先满足需求量最小的13个地区：
最小总运输成本（元）： 3854.370845852836
平均需求满足率： 0.9189165169275


In [50]:
import pandas as pd
import numpy as np

# 读取数据
demand_df = pd.read_excel('附件：数据.xlsx', sheet_name='附件1', index_col=0).squeeze()
distance_df = pd.read_excel('附件：数据.xlsx', sheet_name='附件2', index_col=0)
stock_df = pd.read_excel('附件：数据.xlsx', sheet_name='附件3', index_col=0).squeeze()

# 提取数据
regions = demand_df.index
warehouses = stock_df.index
demand = demand_df.values
distance = distance_df.values
stock = stock_df.values

# 按需求量从小到大排序
sorted_indices = np.argsort(demand)

# 初始化分配矩阵和剩余库存
allocation = np.zeros((len(warehouses), len(regions)))
remaining_stock = stock.copy()

# 贪心算法：按需求从小到大分配
for region_idx in sorted_indices:
    demand_region = demand[region_idx]
    
    # 直接按距离排序仓库（而非单位运输成本，因为单位成本固定）
    warehouse_order = np.argsort(distance[:, region_idx])
    
    # 依次从距离最近的仓库分配
    for warehouse_idx in warehouse_order:
        if demand_region <= 0:
            break
            
        available = remaining_stock[warehouse_idx]
        if available > 0:
            allocated = min(available, demand_region)
            allocation[warehouse_idx, region_idx] = allocated
            remaining_stock[warehouse_idx] -= allocated
            demand_region -= allocated

# 计算结果
satisfaction = np.sum(allocation, axis=0) / demand
average_satisfaction = np.mean(satisfaction)
total_cost = np.sum(allocation * distance * 0.10)

# 转换为DataFrame以便展示
allocation_df = pd.DataFrame(allocation, index=warehouses, columns=regions)
satisfaction_df = pd.DataFrame(satisfaction, index=regions, columns=['满足比例'])

# 输出结果
print('最优配送方案：')
print(allocation_df)
print('\n各地区满足比例：')
print(satisfaction_df)
print(f'\n平均满足比例：{average_satisfaction:.2%}')
print(f'总运输成本：{total_cost:.2f} 元')


最优配送方案：
              石家庄         唐山       秦皇岛         邯郸         邢台   保定        张家口  \
仓库A(玉田)   0.00000  21.000000  9.065831   0.000000   0.000000  0.0   0.000000   
仓库B(辛集)  31.53822   9.175434  0.000000   0.000000   0.000000  0.0  38.169672   
仓库C(南宫)   0.00000   0.000000  0.000000  36.977719  36.911715  0.0   0.000000   

                承德    沧州         廊坊         衡水    北京市   天津市  
仓库A(玉田)  14.959592   0.0  20.309791   0.000000  10.53  6.34  
仓库B(辛集)   0.000000  39.0   0.000000   0.000000   0.00  0.00  
仓库C(南宫)   0.000000   0.0   0.000000  30.385093   0.00  0.00  

各地区满足比例：
         满足比例
石家庄  1.000000
唐山   1.000000
秦皇岛  1.000000
邯郸   1.000000
邢台   1.000000
保定   0.000000
张家口  1.000000
承德   1.000000
沧州   0.977126
廊坊   1.000000
衡水   1.000000
北京市  1.000000
天津市  1.000000

平均满足比例：92.13%
总运输成本：5087.23 元


In [ ]:
import numpy as np
from scipy.optimize import linprog


# 各仓库的化肥实际存量（万吨）
warehouse_stocks = np.array([85, 120, 105])
# 京津冀地区的化肥需求量（万吨）
city_demands = np.array([31.538220487736385, 30.175433907507962, 9.065830953941212, 36.97771906010111, 36.911715151657, 43.38398076202959, 38.16967199494477, 14.959591719715412, 39.91295169443925, 20.309790886538106, 30.38509338138926, 10.53, 6.34])
# 各仓库与配送目标间的公路距离（公里）
distances = np.array([
    [410.2, 63.7, 177.6, 562.2, 516.3, 275.4, 312.3, 162.0, 246.0, 138.0, 376.0, 135.9, 123.2],
    [80.8, 385.3, 530.2, 213.2, 167.3, 149.6, 439.1, 513.4, 168.2, 272.9, 70.6, 292.0, 271.7],
    [124.4, 428.6, 573.6, 150.3, 104.4, 222.5, 518.8, 556.5, 211.5, 315.9, 55.0, 335.1, 315.1]
])
# 运输成本为0.10元/公里/吨
transport_cost = 0.10


for n in range(1, 13):
    # 计算每个地区到各仓库的最小距离，并按最小距离从小到大排序的索引
    min_distances = np.min(distances, axis=0)
    closest_indices = np.argsort(min_distances)[:n]

    # 剩余地区按需求量从小到大排序的索引
    remaining_indices = np.delete(np.arange(len(city_demands)), closest_indices)
    remaining_demands = city_demands[remaining_indices]
    remaining_sorted_indices = remaining_indices[np.argsort(remaining_demands)]

    # 合并两类索引
    priority_indices = np.concatenate((closest_indices, remaining_sorted_indices))

    # 构建目标函数系数，对优先级高的地区赋予高优先级
    c_transport = np.array([dist * transport_cost for sublist in distances for dist in sublist])
    priority_weight = 1000
    c_priority = np.zeros(len(c_transport))
    for index in priority_indices:
        for i in range(len(warehouse_stocks)):
            c_priority[i * len(city_demands)+index] = -priority_weight
    c = c_transport + c_priority

    # 构建不等式约束矩阵A_ub和向量b_ub（每个仓库发货量不超过库存）
    A_ub = []
    b_ub = []
    num_cities = len(city_demands)
    for i in range(len(warehouse_stocks)):
        row = [0] * num_cities * len(warehouse_stocks)
        for j in range(num_cities):
            row[i * num_cities + j] = 1
        A_ub.append(row)
        b_ub.append(warehouse_stocks[i])

    # 构建不等式约束确保每个地区分配量不超过需求量
    for j in range(num_cities):
        row = [0] * num_cities * len(warehouse_stocks)
        for i in range(len(warehouse_stocks)):
            row[i * num_cities + j] = 1
        A_ub.append(row)
        b_ub.append(city_demands[j])

    # 构建等式约束矩阵A_eq和向量b_eq（所有库存都要分配出去）
    A_eq = []
    b_eq = []
    for i in range(len(warehouse_stocks)):
        row = [0] * num_cities * len(warehouse_stocks)
        for j in range(num_cities):
            row[i * num_cities + j] = 1
        A_eq.append(row)
    b_eq = warehouse_stocks

    # 求解线性规划问题
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, method='highs', bounds=(0, None))

    if res.status == 0:
        # 提取结果并输出
        optimal_solution = res.x.reshape(len(warehouse_stocks), len(city_demands))
        print(f'优先满足离仓库最近的{n}个地区：')
        print('最小总运输成本（元）：', np.sum(optimal_solution * distances * transport_cost))

        # 计算各地区需求满足率
        demand_satisfaction = np.sum(optimal_solution, axis=0)/city_demands
        print('平均需求满足率：', np.mean(demand_satisfaction))
    else:
        print(f'线性规划求解失败，状态码: {res.status}')
        if res.status == 1:
            print('超过了迭代次数限制')
        elif res.status == 2:
            print('问题不可行')
        elif res.status == 3:
            print('目标函数无界')
        elif res.status == 4:
            print('算法特定的错误')

优先满足离仓库最近的1个地区：
最小总运输成本（元）： 3854.370845852836
平均需求满足率： 0.9189165169275
优先满足离仓库最近的2个地区：
最小总运输成本（元）： 3854.370845852836
平均需求满足率： 0.9189165169275
优先满足离仓库最近的3个地区：
最小总运输成本（元）： 3854.370845852836
平均需求满足率： 0.9189165169275
优先满足离仓库最近的4个地区：
最小总运输成本（元）： 3854.370845852836
平均需求满足率： 0.9189165169275
优先满足离仓库最近的5个地区：
最小总运输成本（元）： 3854.370845852836
平均需求满足率： 0.9189165169275
优先满足离仓库最近的6个地区：
最小总运输成本（元）： 3854.370845852836
平均需求满足率： 0.9189165169275
优先满足离仓库最近的7个地区：
最小总运输成本（元）： 3854.370845852836
平均需求满足率： 0.9189165169275
优先满足离仓库最近的8个地区：
最小总运输成本（元）： 3854.370845852836
平均需求满足率： 0.9189165169275
优先满足离仓库最近的9个地区：
最小总运输成本（元）： 3854.370845852836
平均需求满足率： 0.9189165169275
优先满足离仓库最近的10个地区：
最小总运输成本（元）： 3854.370845852836
平均需求满足率： 0.9189165169275
优先满足离仓库最近的11个地区：
最小总运输成本（元）： 3854.370845852836
平均需求满足率： 0.9189165169275
优先满足离仓库最近的12个地区：
最小总运输成本（元）： 3854.370845852836
平均需求满足率： 0.9189165169275
